# M27 · Linear & convex optimization

Curriculum · Domain 6 · Optimization

We solve a tiny ads LP by enumerating vertices, then read a shadow price by perturbing the budget. The math shape is $\max c^\top x$ subject to $Ax \le b$ and $x \ge 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(27)

## The Event Ads toy LP

Let $x$ be feed inventory units and $y$ be search inventory units. We maximize expected clicks $5x+4y$ subject to budget $2x+y \le 8$, feed cap $x \le 3$, search cap $y \le 6$, and nonnegativity.

In [ ]:
c = np.array([5.0, 4.0])
A = np.array([[2.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
b = np.array([8.0, 3.0, 6.0])

print("objective coefficients:", c)
print("constraint matrix:")
print(A)

## Enumerate vertices

In two variables, every LP optimum occurs at a polygon vertex. We get candidate vertices by intersecting pairs of active boundaries: three resource constraints plus the two axes.

In [ ]:
boundaries_A = np.vstack([A, np.array([1.0, 0.0]), np.array([0.0, 1.0])])
boundaries_b = np.concatenate([b, np.array([0.0, 0.0])])

vertices = []
for i in range(len(boundaries_b)):
    for j in range(i + 1, len(boundaries_b)):
        M = np.vstack([boundaries_A[i], boundaries_A[j]])
        if abs(np.linalg.det(M)) < 1e-9:
            continue
        point = np.linalg.solve(M, np.array([boundaries_b[i], boundaries_b[j]]))
        if np.all(A @ point <= b + 1e-9) and np.all(point >= -1e-9):
            vertices.append(point)

vertices = np.unique(np.round(np.array(vertices), 10), axis=0)
values = vertices @ c

for point, value in zip(vertices, values):
    print(point, round(value, 2))

## Pick the optimum

The best vertex has the largest value $c^\top x$. The assertion checks the same optimum as the lesson text.

In [ ]:
best_index = int(np.argmax(values))
best_point = vertices[best_index]
best_value = float(values[best_index])

print("best point:", best_point)
print("best value:", best_value)

assert np.allclose(best_point, np.array([1.0, 6.0]))
assert abs(best_value - 29.0) < 1e-9

## Read the shadow price

A shadow price is a marginal value. We re-solve the tiny LP after increasing the budget by $\Delta$ and estimate $\frac{dV}{db}$ from finite differences.

In [ ]:
def solve_budget(budget):
    local_b = np.array([budget, 3.0, 6.0])
    local_vertices = []
    local_boundaries_b = np.concatenate([local_b, np.array([0.0, 0.0])])
    for i in range(len(local_boundaries_b)):
        for j in range(i + 1, len(local_boundaries_b)):
            M = np.vstack([boundaries_A[i], boundaries_A[j]])
            if abs(np.linalg.det(M)) < 1e-9:
                continue
            point = np.linalg.solve(M, np.array([local_boundaries_b[i], local_boundaries_b[j]]))
            if np.all(A @ point <= local_b + 1e-9) and np.all(point >= -1e-9):
                local_vertices.append(point)
    local_vertices = np.unique(np.round(np.array(local_vertices), 10), axis=0)
    local_values = local_vertices @ c
    return float(np.max(local_values))

base_value = solve_budget(8.0)
value_plus = solve_budget(9.0)
shadow_price = value_plus - base_value

print("value at budget 8:", base_value)
print("value at budget 9:", value_plus)
print("shadow price:", shadow_price)

assert abs(shadow_price - 2.5) < 1e-9

## Visualize the feasible polygon

The red point is the optimal allocation. The binding budget and search cap explain why the shadow price is positive.

In [ ]:
order = np.array([0, 1, 4, 3, 2])
polygon = vertices[order]

fig, ax = plt.subplots(figsize=(5, 4))
ax.fill(polygon[:, 0], polygon[:, 1], alpha=0.2, color="#4c78a8")
ax.scatter(vertices[:, 0], vertices[:, 1], color="#4c78a8")
ax.scatter([best_point[0]], [best_point[1]], color="#e45756", s=80)
ax.set_xlabel("feed units x")
ax.set_ylabel("search units y")
ax.set_title("feasible set and optimum")
plt.show()

## Your turn

1. Change the search cap from 6 to 5 and re-solve.
2. Change the feed click coefficient from 5 to 7 and see when the optimum moves.
3. Add a new constraint $x+y \le 7$ and list which constraints bind.

In [ ]:
# Your turn:
